# Catching state drift with `IntegrityGate`

Long agent runs silently corrupt state: a node forgets to set a
required field, a tool mutates a value out of range, a schema
assumption breaks after a refactor and the downstream nodes keep
happily running on garbage. The pathology Paper 4 §4 calls out —
"RAW and GUARDED are completely blind to genome corruption" — maps
straight onto LangGraph state graphs.

`IntegrityGate` runs a list of invariants after each wrapped node and
emits a `langgraph_state_integrity` certificate the first time any
one fails (per thread). A conditional edge can then route the run
to a recovery / escalation path instead of letting corruption
propagate.

In [1]:
import sys
import langgraph
import operon_langgraph_gates as olg

print(f"Python     : {sys.version.split()[0]}")
print(f"langgraph  : {getattr(langgraph, '__version__', 'unknown')}")
print(f"olg        : {olg.__version__}")

Python     : 3.11.15
langgraph  : unknown
olg        : 0.1.0a0


## Baseline — a node silently drops a required field

A three-node pipeline: `fetch` populates state, `transform` mutates
it, `respond` formats the final output. `transform` has a bug — on
this turn it forgets to keep `user_id`. Downstream `respond` sees
`None` and produces a broken response without any error.

In [2]:
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph


class AppState(TypedDict, total=False):
    user_id: str
    budget: int
    answer: str


def fetch(state: AppState) -> AppState:
    return {"user_id": "u_42", "budget": 100}


def bad_transform(state: AppState) -> AppState:
    # bug: drops user_id, exceeds budget
    return {"budget": state["budget"] - 500}


def respond(state: AppState) -> AppState:
    return {"answer": f"hi {state.get('user_id')!r}, balance={state.get('budget')}"}


baseline = StateGraph(AppState)
baseline.add_node("fetch", fetch)
baseline.add_node("transform", bad_transform)
baseline.add_node("respond", respond)
baseline.add_edge(START, "fetch")
baseline.add_edge("fetch", "transform")
baseline.add_edge("transform", "respond")
baseline.add_edge("respond", END)
baseline_app = baseline.compile()

result = baseline_app.invoke({})
print(f"broken answer: {result['answer']!r}")
print(f"budget       : {result['budget']}  (should never go negative)")

broken answer: "hi 'u_42', balance=-400"
budget       : -400  (should never go negative)


## With `IntegrityGate` — invariants fire before corruption spreads

Define two invariants. Wrap the offending node. Add a conditional
edge that routes to `recover` on violation. The gate emits one
certificate at the first failure with a frozen evidence snapshot
listing which invariants passed and which failed.

In [3]:
from operon_langgraph_gates import IntegrityGate


def has_user_id(state: AppState) -> bool:
    return bool(state.get("user_id"))


def budget_not_exceeded(state: AppState) -> bool:
    return state.get("budget", 0) >= 0


def recover(state: AppState) -> AppState:
    return {"answer": "I can't answer right now — please retry."}


gate = IntegrityGate(invariants=[has_user_id, budget_not_exceeded])

gated = StateGraph(AppState)
gated.add_node("fetch", fetch)
gated.add_node("transform", gate.wrap(bad_transform))
gated.add_node("respond", respond)
gated.add_node("recover", recover)
gated.add_edge(START, "fetch")
gated.add_edge("fetch", "transform")
gated.add_conditional_edges(
    "transform",
    gate.edge(forward="respond", break_to="recover"),
)
gated.add_edge("recover", END)
gated.add_edge("respond", END)
gated_app = gated.compile()

result = gated_app.invoke({})
print(f"gated answer: {result['answer']!r}")

gated answer: "I can't answer right now — please retry."


## Inspect the certificate

One `langgraph_state_integrity` cert was emitted on the first
violation. `cert.verify()` replays the evidence: `holds` is
`False` (integrity did not hold), and `evidence` lists which
invariants failed.

In [4]:
assert len(gate.certificates) == 1, "expected exactly one certificate"
cert = gate.certificates[0]
print(f"theorem    : {cert.theorem}")
print(f"source     : {cert.source}")
print(f"conclusion : {cert.conclusion}")

result = cert.verify()
print(f"verify.holds: {result.holds}")
print(f"failed      : {result.evidence.get('failed_invariants')}")
print(f"all results : {result.evidence.get('invariant_results')}")

theorem    : langgraph_state_integrity
source     : operon_langgraph_gates.integrity
conclusion : State-integrity violation: 2 of 2 invariants failed (has_user_id, budget_not_exceeded).
verify.holds: False
failed      : ('has_user_id', 'budget_not_exceeded')
all results : (('has_user_id', False), ('budget_not_exceeded', False))


## Takeaway

Invariants live as ordinary callables in your code; evidence lives
in a signed artifact the first time any one of them fails. Both the
code invariants and the cert survive a restart / audit / replay —
you can feed an old cert back through `cert.verify()` and
re-confirm the finding without re-running the graph.

Paired with `StagnationGate` (see `01_stagnation_breaks_loop.ipynb`),
this gives a LangGraph `StateGraph` two reliability primitives:
**loops stop**, **drift gets caught**, and both surface as
replayable certificates.